# Time Series AIME: a minimal working example

**Tested with `tsaime==0.3.3`.** A small, synthetic example for learning the API and starting a new application. No data downloads, APR files, or `package-v2` are used. The only project dependency is the installed `tsaime` package; NumPy, pandas, Matplotlib and Jupyter are also needed.

We will train a simple two-horizon forecaster, fit an output-to-input explanation, inspect its feature profile and reconstruction, and repeat the fit over trailing time windows. The forecaster here is ordinary linear regression, **not S-Map**; replace it with EDM or another forecasting model in your study. This is an API tutorial, not an additional experiment for the APR paper.

日本語：外部データなしで、予測作成、逆再構成、特徴プロファイル、時間窓ごとの変化を順に確認します。

## Before running

From the source-release directory containing `pyproject.toml`, install and launch:

```bash
python -m pip install -e ".[plot,notebook]"
python -m jupyter lab
```

Select that Python environment as the kernel. Restart the kernel after changing package versions. A wheel installation of `tsaime` 0.3.3 also works; this notebook does not search for or import a local source directory. No GitHub tag or PyPI release is assumed to exist.

Then use **Restart Kernel and Run All Cells**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import tsaime
from tsaime import fit_inverse_operator, RollingVectorTSAIME, RollingVectorTSAIMEConfig

if tsaime.__version__ != "0.3.3":
    raise RuntimeError("Install tsaime 0.3.3 in this kernel's environment, then restart the kernel.")
print("tsaime", tsaime.__version__)

## 1. Make a small time series

Three synthetic input features have different persistence. The measured target combines the first two; the third is an unrelated feature in the data-generating process. The fitted model still receives all three, so finite-sample associations can occur.

Each row is one **forecast origin**. We construct future observed targets at exactly 1 and 6 hours after that origin. These observed targets train the forecaster; they are **not** the output matrix passed to ts-AIME.

日本語：観測された将来の正解値と、モデルが出した予測値を区別します。

In [ ]:
rng = np.random.default_rng(42)
features = ["slow_signal", "fast_signal", "unrelated_feature"]
horizons = np.array([1, 6])
output_names = ["forecast_1h", "forecast_6h"]
n = 1200
states = np.zeros((n + int(horizons.max()), len(features)))
phi = np.array([0.95, 0.60, 0.30])
states[0] = rng.normal(size=3)
for t in range(1, len(states)):
    states[t] = phi * states[t - 1] + np.sqrt(1 - phi**2) * rng.normal(size=3)
observed_target = 0.8 * states[:, 0] + 0.7 * states[:, 1]
X = states[:n]
future_targets = np.column_stack([observed_target[h:h + n] for h in horizons])
dates = pd.date_range("2020-01-01", periods=n, freq="h")
train_end, calibration_end = 600, 900

## 2. Train once, then forecast later origins

The first 600 hours train the forecaster. Origins whose 6-hour target reaches the next period are excluded from training. The next 300 hours calibrate the inverse; the last 300 evaluate its reconstruction. All settings below are fixed for this tutorial, with no tuning on the evaluation period.

The forecast model is then kept fixed. Its two columns are predictions made **at the same origin**, even though they predict different future times.

日本語：予測モデルの学習、逆写像の較正、その後の評価を時間順に分けます。

In [ ]:
origins = np.arange(n)
train_rows = origins + horizons.max() < train_end
# Ordinary least squares is only the tutorial forecaster, not the ts-AIME implementation.
design = np.column_stack([np.ones(n), X])
forward_weights = np.linalg.lstsq(design[train_rows], future_targets[train_rows], rcond=None)[0]
forecasts = design @ forward_weights
calibration = (origins >= train_end) & (origins < calibration_end)
evaluation = origins >= calibration_end
assert np.max(origins[train_rows] + horizons.max()) < train_end
assert not np.any(calibration & evaluation)
print(f"Forecast training: {train_rows.sum()} origins; inverse calibration: {calibration.sum()}; evaluation: {evaluation.sum()}")

## 3. Fit the approximate inverse

The package standardizes each input and output column **using the calibration rows only**, and fits

$$A = S_{XY}(S_{YY}+\lambda I)^{\dagger},\qquad \widehat X_z = Y_z A^\top.$$

Here, $S_{XY}=X_z^\top Y_z/n_c$ and $S_{YY}=Y_z^\top Y_z/n_c$, where $n_c$ is the number of calibration pairs. $A$ has one row per input feature and one column per forecast horizon. `reconstruct()` returns input estimates in the original input units, reusing the calibration means and scales.

日本語：予測ベクトルから入力状態への近似逆写像を、パッケージの関数で推定します。

In [ ]:
inverse = fit_inverse_operator(
    inputs=X[calibration], outputs=forecasts[calibration], ridge=0.01,
)
A = pd.DataFrame(inverse.operator, index=features, columns=output_names)
reconstructed = inverse.reconstruct(forecasts[evaluation])
print("Signed standardized inverse coefficients:")
display(A.round(3))

## 4. Read the feature profile together with reconstruction error

A row's Euclidean norm summarizes the magnitude of its inverse coefficients across the two forecast coordinates. It is a **feature-importance profile of the inverse representation**, not a percentage, a causal effect, or a decomposition of the predicted target.

The error below is measured on later, unused rows. The baseline always reconstructs the calibration mean of each input. A positive RMSE reduction means that the forecasts reconstruct that feature more accurately than this mean-only baseline. A large coefficient norm alone does not establish useful reconstruction; read both columns.

日本語：係数の大きさだけで判断せず、その特徴が後続期間でどれほど再構成できたかも確認します。

In [ ]:
profile = np.linalg.norm(inverse.operator, axis=1)
rmse = np.sqrt(np.mean((X[evaluation] - reconstructed)**2, axis=0))
mean_rmse = np.sqrt(np.mean((X[evaluation] - inverse.x_scaler.mean)**2, axis=0))
summary = pd.DataFrame({
    "inverse_row_norm": profile,
    "reconstruction_RMSE": rmse,
    "calibration_mean_RMSE": mean_rmse,
    "RMSE_reduction_pct": 100 * (1 - rmse / mean_rmse),
}, index=features)
display(summary.round(3))

## 5. Explain one reconstructed input state

For the first evaluation origin, each term is a standardized forecast coordinate multiplied by its inverse coefficient. The two terms sum to the **reconstructed standardized input**, not to the predicted target. Returning to original input units also restores the calibration input mean and scale.

日本語：各予測時間からの項を足すと、入力側の再構成値になります。

In [ ]:
first_forecast_z = inverse.y_scaler.transform(forecasts[evaluation][:1])[0]
terms = pd.DataFrame(inverse.operator * first_forecast_z, index=features, columns=output_names)
terms["sum = reconstructed input z"] = terms.sum(axis=1)
np.testing.assert_allclose(
    terms.iloc[:, -1].to_numpy(), inverse.reconstruct_standardized(forecasts[evaluation][:1])[0],
)
display(terms.round(3))

## 6. Repeat over trailing calendar windows

The rolling API receives a timestamped table with aligned inputs and forecasts. Here each operator uses the preceding 168 hours, with endpoints 48 hours apart. Its window is **(endpoint - 168 hours, endpoint]**: the endpoint observation is included. Such an operator becomes available after that observation arrives; use it for subsequent origins if evaluating out-of-window reconstruction.

The plot below is a descriptive rolling profile, not a held-out error or confidence interval. The separate reconstruction check above uses a fixed inverse calibrated strictly before evaluation. This synthetic process has no designed regime change, so rolling variation is not evidence of a changing physical mechanism.

日本語：窓ごとの係数変化を表示します。この図と、前段の未使用期間での再構成評価は別のものです。

In [ ]:
aligned = pd.DataFrame(X, columns=features)
aligned["Date"] = dates
aligned[output_names] = forecasts
# Exclude the forecaster's training period from inverse explanation.
aligned = aligned.loc[origins >= train_end].reset_index(drop=True)
rolling = RollingVectorTSAIME(RollingVectorTSAIMEConfig(
    window="168h", step="48h", ridge=0.01, min_samples=150,
)).fit(aligned, features=features, outputs=output_names)
rolling_profile = (
    rolling.operators.assign(square=lambda frame: frame["coefficient"]**2)
    .groupby(["endpoint", "feature"])["square"].sum().pow(0.5).unstack("feature")
)
print(f"Estimated {len(rolling.diagnostics)} trailing-window operators.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
summary["inverse_row_norm"].plot.barh(ax=axes[0], color="steelblue")
axes[0].set(xlabel="Inverse coefficient row norm", ylabel="", title="Fixed calibration-window profile")
rolling_profile[features].plot(ax=axes[1])
axes[1].set(xlabel="Calibration-window endpoint", ylabel="Inverse coefficient row norm", title="Trailing 168-hour profiles")
axes[1].legend(fontsize=8)
plt.show()

## Starting your own application

Replace Section 1 with your timestamped input states and target measurements, and Section 2 with your forecasting procedure. Keep Sections 3–6 as the reusable analysis pattern.

- Align every input row and forecast vector by **forecast-origin timestamp**, never by horizon-specific target time. For irregular or missing observations, construct future targets on the actual calendar before dropping incomplete pairs.
- Use predictions produced without training on their future targets. Keep model selection separate from final evaluation; the tutorial fixes its settings and does not demonstrate hyperparameter selection.
- Define the features, forecast horizons, units, calibration length and reconstruction question for your application. Constant or nearly constant columns need separate diagnostics, not an importance interpretation.
- Interpret the coefficient signs and horizon-specific reconstruction terms alongside reconstruction error. Profile magnitudes depend on scaling, output covariance, horizons and regularization; they are not universal feature rankings.
- For a paper, add suitable reconstruction controls, dependence-preserving uncertainty analysis, sensitivity checks and independent data. The small tutorial does not establish scientific novelty or publication readiness.

No change to `src/tsaime/` is needed merely to replace the dataset or forecaster. For S-Map forecasting and a complete air-quality study, see the separate [v9.4 notebook](tsAIME_APR_all_experiments_v9_4.ipynb). It is not a dependency of this example.

日本語：新しい応用ではデータと予測器を置き換え、同じパッケージを利用できます。研究固有の処理や図表は、その研究のノートブックに置いてください。

Software use follows the release's [LICENSE.txt](../LICENSE.txt). Record `tsaime` 0.3.3 and the exact source revision or archived software version used in your study; see [CITATION.cff](../CITATION.cff).